# L8: Media Agent Build with AI (previous lesson)

This is the agent built in the past lesson using GeminiCLI.

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

In [ ]:
import requests
import datetime
from google import genai
from google.genai import types
from google.adk.agents.llm_agent import LlmAgent
import PIL.Image

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access  <code>skills</code>, <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>
</div>

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different run results:</b> The output generated by AI models, and agents can vary with each execution due to their dynamic, probabilistic nature. Don't be surprised if your results differ from those shown in the video.</p>

In [ ]:
# Setup Logging
LOG_FILE = "infographic_agent.log"

def log_step(message: str):
    timestamp = datetime.datetime.now().isoformat()
    with open(LOG_FILE, "a") as f:
        f.write(f"[{timestamp}] {message}\n")

# Use this auth code for DeepLearning env
import os
from helper import authenticate

credentials, project_id = authenticate()
client = genai.Client(
    project=project_id,
    location="global",
    credentials=credentials,
    http_options=types.HttpOptions(
         base_url=os.getenv("GOOGLE_VERTEX_BASE_URL")
    )
)
# ---------------------------------------------------------

def fetch_blog_content(url: str) -> str:
    """Fetches the text content of a blog post."""
    log_step(f"Fetching blog content from: {url}")
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        # Simple text extraction (heuristic)
        content = response.text
        log_step("Successfully fetched blog content.")
        return content[:5000] # Limit content for prompt efficiency
    except Exception as e:
        log_step(f"Error fetching blog content: {e}")
        return f"Error: {e}"

def generate_infographic(blog_content: str, feedback: str = "") -> str:
    """Generates an infographic using Nano Banana."""
    log_step("Generating infographic...")
    prompt = f"Create a professional infographic based on this blog content: {blog_content}."
    if feedback:
        log_step(f"Applying feedback for regeneration: {feedback}")
        prompt += f" Please fix the following issues from the previous attempt: {feedback}"
    
    try:
        response = client.models.generate_content(
            model="gemini-3.1-flash-image-preview",
            contents=prompt,
            config=types.GenerateContentConfig(
                response_modalities=["IMAGE"],
            )
        )
        
        image_part = next(p for p in response.candidates[0].content.parts if p.inline_data)
        image_filename = f"infographic_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
        with open(image_filename, "wb") as f:
            f.write(image_part.inline_data.data)
        
        log_step(f"Infographic saved as {image_filename}")
        return image_filename
    except Exception as e:
        log_step(f"Error generating infographic: {e}")
        return f"Error: {e}"

def evaluate_infographic(image_path: str, blog_content: str) -> str:
    """Evaluates the image for factual accuracy, spelling, and aesthetics."""
    log_step(f"Evaluating infographic: {image_path}")
    try:
        img = PIL.Image.open(image_path)
        
        evaluation_prompt = (
            "Analyze this infographic against the following blog content for:\n"
            "1. Factual Accuracy: Does it correctly represent the information?\n"
            "2. Spelling: Are there any typos or spelling errors in the text?\n"
            "3. Aesthetic Alignment: Does the style match a professional blog?\n\n"
            "If it fails any criteria, provide specific feedback for improvement. "
            "If it passes all criteria, respond ONLY with 'PASS'.\n\n"
            f"Blog Content: {blog_content}"
        )
        
        response = client.models.generate_content(
            model="gemini-3-flash-preview",
            contents=[img, evaluation_prompt]
        )
        
        result = response.text.strip()
        log_step(f"Evaluation result: {result}")
        return result
    except Exception as e:
        log_step(f"Error evaluating infographic: {e}")
        return f"Evaluation Error: {e}"

def infographic_workflow(url: str) -> str:
    """The main workflow for generating and validating the infographic."""
    log_step(f"Starting infographic workflow for URL: {url}")
    
    content = fetch_blog_content(url)
    if "Error" in content:
        return content
    
    max_attempts = 3
    feedback = ""
    
    for attempt in range(max_attempts):
        log_step(f"Attempt {attempt + 1} of {max_attempts}")
        image_path = generate_infographic(content, feedback)
        
        if "Error" in image_path:
            return image_path
            
        eval_result = evaluate_infographic(image_path, content)
        
        if eval_result == "PASS":
            log_step("Infographic passed evaluation.")
            return f"Success! Infographic saved at: {image_path}"
        else:
            feedback = eval_result
            log_step(f"Regenerating due to feedback: {feedback}")
            
    log_step("Reached maximum attempts. Returning last version.")
    return f"Workflow completed with feedback (check log): {image_path}"

from google.adk.models import Gemini
class DLAIGemini(Gemini):
    _client: genai.Client = None  # class-level cache
    
    @property
    def api_client(self) -> genai.Client:
        if self._client is None:
            self._client = genai.Client(
                project=project_id,
                location="global",
                credentials=credentials,
                http_options=types.HttpOptions(
                    base_url=os.getenv("GOOGLE_VERTEX_BASE_URL")
                )
            )
        return self._client
        
# Define the ADK Agent
root_agent = LlmAgent(
    name="InfographicAgent",
    model=DLAIGemini(model="gemini-3-flash-preview"),
    instruction=(
        "You are an expert at creating infographics from blog posts. "
        "Use the tools provided to fetch content, generate an image, and validate it. "
        "Always log your progress."
    ),
    tools=[infographic_workflow]
)

In [ ]:
from google.adk.runners import InMemoryRunner

BLOG = "https://cloud.google.com/blog/products/ai-machine-learning/ultimate-prompting-guide-for-lyria-3-pro"

runner = InMemoryRunner(agent=root_agent, app_name="image_agent")

session = await runner.session_service.create_session(
    app_name="image_agent", user_id="user"
)

user_message = types.Content(
    role="user",
    parts=[types.Part(text=(
        f"Create an infographic from this blog: {BLOG}"
    ))]
)

async for event in runner.run_async(
    user_id="user",
    session_id=session.id,
    new_message=user_message,
):
    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                print(part.text)
            if part.function_call:
                print(f"\u2192 Calling: {part.function_call.name}")

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access  the generated image:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.
</div>